# Forecast diagnostic: walkthrough

This notebook runs the toolkit on the synthetic sample data and shows each step of the pipeline:

**clean → validate → reconcile → diagnose → act → summarise**

Everything comes from `forecast_diagnostic.py`; the settings come from `config.json`. To run it on your own
export, point `config.json` at your two CSVs and run this notebook again.

In [1]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import forecast_diagnostic as fd

res = fd.run(ROOT / "config.json", use_ai=False, export_outputs=False)
F = res.facts
print(f"{res.cfg['region_name']}  |  snapshot {F['snapshot_date']}  |  {F['days_left_in_quarter']} days to quarter end")
print(f"{len(res.opp_raw)} rows in, {len(res.opp)} after cleaning  |  FX rates inferred: {res.fx}")

North America  |  snapshot 2027-04-16  |  14 days to quarter end
479 rows in, 473 after cleaning  |  FX rates inferred: {'BRL': 0.19, 'CAD': 0.73, 'MXN': 0.058, 'USD': 1.0}


## 1. Clean and validate

Every check that found something, with the rows and dollars it touched. Fixes the data can answer itself
(duplicates, currency, country codes, deal age) are applied and logged; anything that needs a human decision
is flagged instead of guessed.

In [2]:
res.dq_log[["check_id", "check", "severity", "action", "rows_affected", "usd_affected"]]

,check_id,check,severity,action,rows_affected,usd_affected
0,DQ01,Exact duplicate opportunity rows,High,Removed,6,112559.0
1,DQ03,amount_usd not converted at the currency's rate,High,Corrected (recomputed),9,-2191800.0
2,DQ09,Quota assigned to vacant seats (Open Req),High,Flagged,2,760000.0
3,DQ10,Open deals owned by a vacant seat,High,Flagged - reassign,5,287226.0
4,DQ17,Open deal with a close date already in the past,High,Flagged - confirm or push close date,147,6459576.0
5,DQ05,Country spelled out instead of roster ISO code,Medium,Corrected (mapped),34,1164434.0
6,DQ12,Forecast category blank,Medium,Flagged,24,1066189.0
7,DQ13,'Commit' on an early-stage deal,Medium,Flagged - not used in forecast,10,421965.0
8,DQ14,Late-stage deal marked Omitted / blank,Medium,Flagged - not used in forecast,35,1626849.0
9,DQ15,age_days inconsistent with dates (incl. negati...,Medium,Corrected (recomputed from created_date),94,4432784.0


## 2. Which roll-up reproduces the reported number?

The self-reported `forecast_category` and the stage-based calculation give very different answers. Only one
of them reconciles with the number leadership is quoting, and only after cleaning.

In [3]:
res.reconciliation[["method", "raw_pct_of_quota", "clean_pct_of_quota", "matches_reported"]]

,method,raw_pct_of_quota,clean_pct_of_quota,matches_reported
0,Closed Won only,0.399636,0.392987,False
1,Won + open 'Commit' (category),0.587379,0.580160,False
2,Won + open 'Commit' + 'Best Case' (category),0.759048,0.750774,False
3,Won + open pipeline x win probability (stage-b...,0.985488,0.890893,True


Is the category worth trusting? If it meant anything, `Commit` deals would carry a much higher win
probability than `Pipeline` deals.

In [4]:
res.category_audit[["deals", "open_usd", "avg_win_prob", "share_in_early_stage", "share_past_due"]].round(2)

,deals,open_usd,avg_win_prob,share_in_early_stage,share_past_due
forecast_category_clean,,,,,
Commit,44,2035505.0,0.50,0.23,0.64
Best Case,46,1855425.0,0.46,0.20,0.54
Pipeline,126,6276900.0,0.42,0.37,0.54
Omitted,30,1741216.0,0.36,0.50,0.47
(blank),24,1066189.0,0.49,0.17,0.50


## 3. The forecast, and how fragile it is

Forecast = Closed Won + every open deal × its win probability. The scenarios show what happens if the deals
that already missed their close date are worth less than the pipeline claims.

In [5]:
res.scenarios[["scenario", "forecast_usd", "pct_of_quota", "gap_usd"]]

,scenario,forecast_usd,pct_of_quota,gap_usd
0,Stage-weighted forecast (the reconciled number),9688462.190,0.890893,-1186537.810
1,Risk-adjusted: past-due deals at 50% of stated...,8233609.055,0.757113,-2641390.945
2,Downside: past-due deals slip out entirely,6778755.920,0.623334,-4096244.080
3,Rep call: Won + 'Commit' category,6309239.000,0.580160,-4565761.000


## 4. Where the gap is

Quota comes from the roster, the forecast from the deals. Vacant seats are split out from active reps,
because a coverage gap and a performance gap need different fixes.

In [6]:
cols = ["country", "segment", "quota_usd", "vacant_quota_usd", "forecast_usd", "attainment", "gap_usd",
        "share_of_gap", "coverage_of_remaining", "past_due_share_of_weighted", "win_rate_usd"]
res.by_manager[cols].round(3)

,country,segment,quota_usd,vacant_quota_usd,forecast_usd,attainment,gap_usd,share_of_gap,coverage_of_remaining,past_due_share_of_weighted,win_rate_usd
manager_name,,,,,,,,,,,
Theo Vandermeer,CA,Commercial,1900000,760000,1353406.29,0.712,-546593.71,0.461,1.364,0.559,0.870
Beatriz Salgado,BR,Enterprise,2480000,0,2184384.44,0.881,-295615.56,0.249,1.898,0.519,0.802
Carmen Dorsey,US,Enterprise,3000000,0,2851199.49,0.950,-148800.51,0.125,2.219,0.541,0.724
Rafael Ibarra,MX,Public Sector,1170000,0,1080143.59,0.923,-89856.41,0.076,2.218,0.962,0.884
Nadine Lockhart,CA,SMB,1125000,0,1062718.82,0.945,-62281.18,0.052,2.323,0.264,0.870
Harriet Okonjo,US,SMB,1200000,0,1156609.56,0.964,-43390.44,0.037,2.033,0.413,0.781


In [7]:
res.bridge   # quota -> forecast, one row per team, vacant seats on their own line

,label,manager_name,seat,quota_usd,forecast_usd,gap_usd
6,Theo Vandermeer (CA Commercial) - vacant seats,Theo Vandermeer,vacant seats,760000,251711.89,-508288.11
0,Beatriz Salgado (BR Enterprise),Beatriz Salgado,active reps,2480000,2184384.44,-295615.56
1,Carmen Dorsey (US Enterprise),Carmen Dorsey,active reps,3000000,2851199.49,-148800.51
4,Rafael Ibarra (MX Public Sector),Rafael Ibarra,active reps,1170000,1080143.59,-89856.41
3,Nadine Lockhart (CA SMB),Nadine Lockhart,active reps,1125000,1062718.82,-62281.18
2,Harriet Okonjo (US SMB),Harriet Okonjo,active reps,1200000,1156609.56,-43390.44
5,Theo Vandermeer (CA Commercial) - active reps,Theo Vandermeer,active reps,1140000,1101694.40,-38305.60


## 5. Pipeline health: stage, cohort and hygiene

In [8]:
res.health["by_stage"]

,deals,open_usd,weighted_usd,avg_win_prob,past_due_deals,past_due_weighted_usd,median_age_days
pipeline_stage,,,,,,,
Prospecting,27,1311267.0,220763.26,0.154074,9,31744.23,37.0
Qualification,58,2896771.0,877968.93,0.313276,31,365087.16,92.5
Proposal/Price Quote,113,5793291.0,2394324.23,0.425044,63,1423841.56,106.0
Negotiation/Review,72,2973906.0,1921671.77,0.659583,44,1089033.32,129.0


In [9]:
res.health["by_cohort"]   # deals created before the quarter vs during it

,deals,open_usd,weighted_usd,past_due_deals
cohort,,,,
Created before quarter,147,6459576.0,2909706.27,147
Created in quarter,123,6515659.0,2505021.92,0


## 6. Lists a manager can act on this week

In [10]:
show = ["opp_id", "account_name", "rep_name", "manager_name", "pipeline_stage", "close_date",
        "days_past_due", "amount_usd", "win_probability", "weighted_open_usd"]
res.actions["past_due"][show].head(10)

,opp_id,account_name,rep_name,manager_name,pipeline_stage,close_date,days_past_due,amount_usd,win_probability,weighted_open_usd
164,006Pg00000bRWi8QAG,Pinehollow Manufacturing,Otavio Rennard,Beatriz Salgado,Proposal/Price Quote,2027-03-30,17,264144.0,0.41,108299.04
98,006Pg00000Wir2PQAG,Dunbarton Systems,Dexter Moyo,Carmen Dorsey,Negotiation/Review,2027-04-10,6,136422.0,0.72,98223.84
344,006Pg00000tnoghQAG,Pinehollow Utilities,Priscilla Vance,Carmen Dorsey,Proposal/Price Quote,2027-02-22,53,144920.0,0.56,81155.20
58,006Pg00000xX3KKQAG,Fairhaven Health Group,Nayara Quintal,Beatriz Salgado,Negotiation/Review,2027-02-22,53,147644.0,0.53,78251.32
10,006Pg000004utzVQAG,Pinehollow Broadcasting,Emmett Sorrell,Carmen Dorsey,Proposal/Price Quote,2027-03-17,30,143328.0,0.52,74530.56
49,006Pg00000VAG3eQAG,Stonebridge Credit Union,Otavio Rennard,Beatriz Salgado,Qualification,2027-02-09,66,160903.0,0.44,70797.32
248,006Pg00000hiipSQAG,Rivendale Telecom,Larissa Pequeno,Beatriz Salgado,Negotiation/Review,2027-02-22,53,104655.0,0.64,66979.20
113,006Pg00000VM9Y4QAG,Brightmoor Advisors,Emmett Sorrell,Carmen Dorsey,Proposal/Price Quote,2027-02-06,69,238974.0,0.28,66912.72
92,006Pg000001zzZnQAG,Quarry Hill Credit Union,Emmett Sorrell,Carmen Dorsey,Proposal/Price Quote,2027-02-05,70,205307.0,0.31,63645.17
305,006Pg00000KTd8NQAG,Granite Peak Grocers,Caio Brandao,Beatriz Salgado,Proposal/Price Quote,2027-03-03,44,104544.0,0.54,56453.76


In [11]:
res.actions["orphaned_deals"][show]     # open deals owned by a seat nobody sits in

,opp_id,account_name,rep_name,manager_name,pipeline_stage,close_date,days_past_due,amount_usd,win_probability,weighted_open_usd
76,006Pg00000y5dBpQAG,Quarry Hill Advisors,OPEN REQ (Marcus Iwu backfill pending),Theo Vandermeer,Proposal/Price Quote,2027-02-15,60,101414.0,0.39,39551.46
363,006Pg00000m0359QAG,Cascadia Grocers,OPEN REQ (Marcus Iwu backfill pending),Theo Vandermeer,Proposal/Price Quote,2027-03-29,18,75017.0,0.46,34507.82
54,006Pg00000LFWGOQAG,Quarry Hill Manufacturing,OPEN REQ (Delia Santoro backfill pending),Theo Vandermeer,Proposal/Price Quote,2027-02-24,51,52684.0,0.57,30029.88
351,006Pg000006EbPSQAG,Cascadia Biosciences,OPEN REQ (Delia Santoro backfill pending),Theo Vandermeer,Negotiation/Review,2027-04-19,0,35381.0,0.63,22290.03
184,006Pg000004AfJLQAG,Pinehollow Systems,OPEN REQ (Delia Santoro backfill pending),Theo Vandermeer,Prospecting,2027-02-06,69,22730.0,0.19,4318.70


## 7. The summary

`template_summary()` writes the briefing from the facts pack with no AI involved. With an API key set,
`fd.draft_summary(F, res.cfg, use_ai=True)` asks Claude to write it instead, then checks every number it
wrote against the same facts pack and falls back to this template if anything cannot be traced.

In [12]:
from IPython.display import Markdown

Markdown(fd.template_summary(F))

## North America forecast summary - snapshot 2027-04-16 (14 days to quarter end)

**Headline.** North America is forecasting $9.69M against $10.88M quota (89.1%), a gap of $1.19M. 71% of the gap sits in two teams: Theo Vandermeer (CA Commercial) at 71% and Beatriz Salgado (BR Enterprise) at 88%.

**What is driving the gap**
- **Empty seats:** 2 vacant seats carry $760K of quota with only $252K forecast against it (43% of the gap). 5 open deals ($287K) have no active owner.
- **Stale pipeline:** 147 of 270 open deals are past their close date, holding $2.91M (54%) of weighted pipeline. Worst: Rafael Ibarra (MX Public Sector), 96% past due.
- **Losses:** Carmen Dorsey (US Enterprise) lost the most this quarter ($491K).
- **Forecast category not reliable:** 10 'Commit' deals are still early stage and 35 late-stage deals are Omitted or blank, so the call is built from stage and win probability instead.

**Risk.** If past-due deals close at half their stated odds, the region lands at 76% ($8.23M). Call the quarter as a range: 76% to 89%.

**Recommended actions**
- **Next 14 days:** reassign the 5 deals with no owner ($287K) today; review the 44 past-due Negotiation/Review deals ($1.09M weighted): confirm a dated next step or move them out of the quarter; call the quarter as a range, 76% to 89%, not on the Commit category.
- **By day 30:** backfill the vacant seats and cover their territories in the meantime; no open deal may carry a past close date (weekly automatic flag); set entry criteria for 'Commit' (start with the 10 early-stage Commit deals).
- **By day 60:** loss review with Carmen Dorsey's team ($491K lost this quarter); coaching plans for reps below 90% in both prior quarters (Larissa Pequeno, Bruno Cavazos).

**Data confidence.** 14 data checks fired (5 high severity). 6 duplicate rows removed, 9 currency conversions corrected, 34 country codes fixed before any number was calculated.

In [13]:
check = fd.verify_numbers(fd.template_summary(F), F)
print(f"{int(check['verified'].sum())}/{len(check)} numbers in the summary trace back to the facts pack")

35/35 numbers in the summary trace back to the facts pack


## 8. Run it on your own data

1. Export your opportunities and rep roster as CSV, with the columns listed in the README.
2. Point `opportunity_file` and `roster_file` in `config.json` at them (stage names, thresholds, FX and
   country spellings live there too).
3. Run this notebook again, or `python forecast_diagnostic.py`, which also writes the Excel workbook,
   the charts and the summary into `outputs/`.